# Notebook 22 — Leakage-safe hidden-state reconstruction

## 22.1 Purpose and application research question

The empirical application asks whether hidden realised-implied response information discovered by the event-response and hidden-state pipeline contains incremental economic value for USD/JPY option trading. The dissertation progresses from synthetic recovery, through empirical realised/implied dependence and known-event response weights, to flexible-model robustness and hidden abnormal-response states. **N22 does not test trading performance**: it constructs historical state information for downstream work only.

## 22.2 Historical reconstruction disclosure

This is a **retrospective chronological reconstruction conditional on the final fixed specification**. Final N19 structures, lags, hyperparameters, fixed neural epochs, N17 floors, alpha, event universe/timing, empirical-tail and consensus rules were selected retrospectively with later pre-TEST information and are held fixed. Within each historical stage, fitted parameters, scalers and training observations use only preceding information available under that stage's boundary. This is not presented as a prospective or live backtest, and it does not claim that a 2008 observer could have selected the final detector design. It asks how that final fixed methodology behaves when refitted chronologically; the reconstructed TEST classifications are expected to reproduce the final detector assignments.

In [1]:
from pathlib import Path
from copy import deepcopy
import math, random, warnings
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore', message='enable_nested_tensor'); torch.set_num_threads(1)
PROJECT_MARKER = "00_model_specification_and_core_simulator.ipynb"

def find_project_root(start=None):
    """Find the project from the current directory or one of its parents."""
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Data").is_dir() and (candidate / PROJECT_MARKER).is_file():
            return candidate
    raise RuntimeError(
        "Could not locate the FX volatility project root. "
        "Start the notebook from the project directory or one of its subdirectories; "
        f"expected both Data/ and {PROJECT_MARKER}."
    )

ROOT=find_project_root(); PROCESSED=ROOT/'Data'/'processed'
panel=pd.read_csv(PROCESSED/'16_empirical_analysis_panel.csv',parse_dates=['model_day']).sort_values('model_day').reset_index(drop=True)
panel['Q']=pd.to_numeric(panel['squared_return']); panel['R']=pd.to_numeric(panel['realised_variance_ann_252']); panel['I']=pd.to_numeric(panel['iv_model_var'])
# Exact N19 convention: construct canonical Q/R/I lags on the intact chronology before any block filter.
for _state in ('Q','R','I'):
 for _lag in range(1,13): panel[f'{_state}_lag{_lag}']=panel[_state].shift(_lag)
FLEXIBLE_MODELS=('RF','GB','MLP','TRANSFORMER'); PRIMARY_ALPHA=.05; MIN_MODEL_SUPPORT=3
FLOORS={'R':.0003202993332501,'I':.0015767239436135}; SEEDS=(19,119,219)
SPECS={('RF','R'):('R|I',1,'RF_01',None),('GB','R'):('R|I',2,'GB_02',None),('MLP','R'):('R|I',1,'MLP_04',143),('TRANSFORMER','R'):('R|I',1,'TR_01',52),('RF','I'):('I',2,'RF_01',None),('GB','I'):('I',2,'GB_02',None),('MLP','I'):('I|Q',4,'MLP_03',77),('TRANSFORMER','I'):('I|Q|R',5,'TR_03',19)}
RF_GRID={'RF_01':{'max_depth':8,'min_samples_leaf':5}}; GB_GRID={'GB_02':{'n_estimators':200,'learning_rate':.05}}
MLP_GRID={'MLP_03':([32,16],1e-3),'MLP_04':([32,16],3e-4)}; TR_GRID={'TR_01':((16,2,32),1e-3),'TR_03':((24,4,48),1e-3)}
RF_FIXED={'n_estimators':500,'max_features':.5,'criterion':'squared_error','bootstrap':True,'random_state':19,'n_jobs':-1}; GB_FIXED={'max_depth':2,'min_samples_leaf':10,'subsample':1.,'loss':'squared_error','random_state':19}
# Final N19 configuration is frozen by the final N19 notebook definition and this N22 precommitment; no selection export is reused as an estimator or selector.
n19_specification_provenance='final N19 frozen code definitions: structures, lags, selected configuration IDs, fixed neural epochs, seeds, scaling, and admissibility floors'
assert panel.model_day.is_unique and panel.model_day.is_monotonic_increasing
# Frozen N21 pairing chronology: observed canonical model-day rows, never calendar arithmetic.
panel['canonical_index']=np.arange(len(panel),dtype=int)
panel['previous_model_day']=panel['model_day'].shift(1)
panel['previous_sample_split']=panel['sample_split'].shift(1)

In [2]:
# Surgical-pass baseline: loaded before any N22 export is overwritten in this fresh run.
def _read_n22_baseline(name, dates=()):
    path=PROCESSED/name
    return pd.read_csv(path, parse_dates=list(dates)) if path.exists() else None
pre_surgical_snapshot={
    'adequacy':_read_n22_baseline('22_reference_adequacy_audit.csv'),
    'thresholds':_read_n22_baseline('22_stage_consensus_thresholds.csv'),
    'stage_summary':_read_n22_baseline('22_stage_state_summary.csv'),
    'application':_read_n22_baseline('22_application_state_panel.csv',('model_day','previous_model_day')),
    'episodes':_read_n22_baseline('22_prospective_episodes.csv',('start_day','end_day','peak_day')),
}
assert all(value is not None for value in pre_surgical_snapshot.values()), 'Expected pre-surgical N22 outputs are unavailable.'
print('Surgical invariance baseline captured from current executed N22 exports.')


Surgical invariance baseline captured from current executed N22 exports.


## 22.3–22.6 Frozen specifications and historical ordinary forecasts

In [3]:
def cols(parents,lag): return [f'{x}_lag{i}' for i in range(lag,0,-1) for x in parents]
def frame(target,days,parents,lag):
 c=[target,*cols(parents,lag)]; return panel.loc[panel.model_day.isin(days)&np.isfinite(panel[c].to_numpy(float)).all(1)].copy()
def seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s)
def scale_fit(x,target,parents,lag):
 d={};
 for z in parents:
  v=x[[f'{z}_lag{i}' for i in range(1,lag+1)]].to_numpy(float).ravel();d[z]=(v.mean(),v.std())
 y=x[target].to_numpy(float);d['Y']=(y.mean(),y.std());return d
def sx(x,c,d):
 a=x[c].to_numpy(float).copy()
 for j,k in enumerate(c):m,s=d[k.split('_lag')[0]];a[:,j]=(a[:,j]-m)/s
 return a
class MLP(nn.Module):
 def __init__(self,n,h):
  super().__init__();a=[]
  for z in h:a += [nn.Linear(n,z),nn.ReLU(),nn.Dropout(.1)];n=z
  a += [nn.Linear(n,1)];self.n=nn.Sequential(*a)
 def forward(self,x):return self.n(x).squeeze(-1)
class TR(nn.Module):
 def __init__(self,p,l,d,h,f):
  super().__init__();self.e=nn.Linear(p,d);pos=torch.arange(l).unsqueeze(1);div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.)/d));pe=torch.zeros(l,d);pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div);self.register_buffer('pe',pe.unsqueeze(0));self.register_buffer('mask',torch.triu(torch.ones(l,l,dtype=torch.bool),1));self.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,h,f,dropout=.1,activation='gelu',batch_first=True),1);self.o=nn.Linear(d,1)
 def forward(self,x):return self.o(self.enc(self.e(x)+self.pe,mask=self.mask)[:,-1]).squeeze(-1)
def predict(model,target,cal,pred):
 ps,lag,cfg,epochs=SPECS[(model,target)];parents=tuple(ps.split('|'));c=cols(parents,lag)
 if model=='RF':
  e=RandomForestRegressor(**RF_FIXED,**RF_GRID[cfg]).fit(cal[c],cal[target]);return e.predict(pred[c])
 if model=='GB':
  e=GradientBoostingRegressor(**GB_FIXED,**GB_GRID[cfg]).fit(cal[c],cal[target]);return e.predict(pred[c])
 out=[]
 for z in SEEDS:
  seed(z);d=scale_fit(cal,target,parents,lag);x=sx(cal,c,d);y=(cal[target].to_numpy(float)-d['Y'][0])/d['Y'][1];t=torch.tensor(x,dtype=torch.float32);t=t.reshape(-1,lag,len(parents)) if model=='TRANSFORMER' else t;n,lr=(MLP(len(c),MLP_GRID[cfg][0]),MLP_GRID[cfg][1]) if model=='MLP' else (TR(len(parents),lag,*TR_GRID[cfg][0]),TR_GRID[cfg][1]);opt=torch.optim.AdamW(n.parameters(),lr=lr,weight_decay=1e-4);dl=DataLoader(TensorDataset(t,torch.tensor(y,dtype=torch.float32)),batch_size=64,shuffle=True)
  for _ in range(epochs):
   for bx,by in dl:
    opt.zero_grad();loss=nn.MSELoss()(n(bx),by);loss.backward()
    # Frozen N19 clipping and optimizer step are batch-level operations.
    if model=='TRANSFORMER':torch.nn.utils.clip_grad_norm_(n.parameters(),1.)
    opt.step()
  xp=sx(pred,c,d);xp=torch.tensor(xp,dtype=torch.float32);xp=xp.reshape(-1,lag,len(parents)) if model=='TRANSFORMER' else xp
  with torch.no_grad():out.append(n(xp).numpy()*d['Y'][1]+d['Y'][0])
 return np.mean(out,axis=0)
def forecasts(cal_days,pred_days,stage):
 r=[]
 for m in FLEXIBLE_MODELS:
  for t in ('R','I'):
   ps,l,_,_=SPECS[(m,t)];pa=tuple(ps.split('|'));cal=frame(t,cal_days,pa,l);pr=frame(t,pred_days,pa,l);raw=predict(m,t,cal,pr);adm=np.maximum(raw,FLOORS[t])
   r += [{'model_day':d,'sample_split':stage,'target':t,'model':m,'actual':float(a),'raw_forecast':float(x),'admissible_forecast':float(y),'ordinary_response_ratio':float(a/y),'ordinary_flag':True,'floor_activated':bool(x<FLOORS[t])} for d,a,x,y in zip(pr.model_day,pr[t],raw,adm)]
 return pd.DataFrame(r)

## 22.7–22.8 Candidate chronology and Phase A adequacy audit

In [4]:
# 22.4–22.8: canonical chronology, historical ordinary responses, and the pre-state gate.
A = panel.loc[panel.model_day.between('2003-05-05','2007-12-19'),'model_day']
B = panel.loc[panel.model_day.between('2007-12-20','2012-08-07'),'model_day']
C = panel.loc[panel.model_day.between('2012-08-08','2017-03-27'),'model_day']
V = panel.loc[panel.sample_split.eq('validation'),'model_day']
T = panel.loc[panel.sample_split.eq('test'),'model_day']
assert (len(A),len(B),len(C),len(V),len(T)) == (1208,1209,1209,1209,1209)
assert (A.min(),A.max(),B.min(),B.max(),C.min(),C.max()) == tuple(pd.Timestamp(x) for x in ['2003-05-05','2007-12-19','2007-12-20','2012-08-07','2012-08-08','2017-03-27'])

panel['canonical_sample_split']=panel['sample_split']
panel.loc[panel.model_day.isin(B),'sample_split']='B'
panel.loc[panel.model_day.isin(C),'sample_split']='C'
panel['previous_sample_split']=panel['sample_split'].shift(1)
# Expanding parameter-estimation history; these are ordinary/event-unaware forecasts only.
historical_B = forecasts(A,B,'B')
historical_C = forecasts(pd.concat([A,B]),C,'C')

# Exact final N20 ordinary inputs that N21 loads for VALIDATION and TEST.
canonical = pd.read_csv(PROCESSED/'20_api_ordinary_response_ratios.csv',parse_dates=['model_day'])
canonical = canonical.loc[canonical.model.isin(FLEXIBLE_MODELS)].copy()
canonical['ordinary_response_ratio'] = pd.to_numeric(canonical['ordinary_response_ratio'],errors='coerce')
canonical_cols = ['model_day','sample_split','target','model','actual','raw_forecast','admissible_forecast','ordinary_response_ratio','ordinary_flag','floor_activated']
ratios = pd.concat([historical_B,historical_C,canonical.loc[canonical.sample_split.isin(['validation','test']),canonical_cols]],ignore_index=True)
assert ratios.loc[ratios.sample_split.isin(['validation','test']),'ordinary_response_ratio'].notna().all()

# Final N18 target-specific six-family event timing.  Retain N20 flags unchanged for V/T.
events = pd.read_csv(PROCESSED/'18_target_specific_event_mapping.csv',parse_dates=['target_model_day'])
events = events.loc[events.family.isin(['FOMC','US Employment','US CPI','BoJ','JPY National CPI','JPY GDP'])].copy()
for target in ('R','I'):
    event_days = set(events.loc[events.target.eq(target),'target_model_day'])
    historical_target = ratios.sample_split.isin(['B','C']) & ratios.target.eq(target)
    ratios.loc[historical_target,'ordinary_flag'] = ~ratios.loc[historical_target,'model_day'].isin(event_days)
ratios['ordinary_flag'] = ratios['ordinary_flag'].fillna(False).astype(bool)
ratios['omega'] = ratios['ordinary_response_ratio']
ratios['log_omega'] = np.where(np.isfinite(ratios.omega) & ratios.omega.gt(0),np.log(ratios.omega),np.nan)
ratios['log_eligible'] = ratios.log_omega.notna()

# Phase A: ordinary-reference counts only.  No state labels exist above this line.
model_counts = (ratios.loc[ratios.sample_split.isin(['B','C']) & ratios.ordinary_flag & ratios.log_eligible]
    .groupby(['sample_split','target','model']).size().rename('n_model_reference').reset_index())
n21_model_comparators = pd.read_csv(PROCESSED/'21_empirical_cdf_reference_summary.csv')
n21_model_comparators = n21_model_comparators.rename(columns={'n_reference':'n21_validation_model_reference'})[['target','model','n21_validation_model_reference']]
model_counts = model_counts.merge(n21_model_comparators,on=['target','model'],how='left',validate='many_to_one')
model_counts['ratio_to_n21_validation_comparator'] = model_counts.n_model_reference / model_counts.n21_validation_model_reference
assert model_counts.n21_validation_model_reference.notna().all()

cons_counts = (ratios.loc[ratios.sample_split.isin(['B','C']) & ratios.ordinary_flag & ratios.log_eligible]
    .groupby(['sample_split','target','model_day']).model.nunique().ge(MIN_MODEL_SUPPORT)
    .groupby(level=[0,1]).sum().rename('n_consensus_reference').reset_index())
n21_consensus_comparators = pd.read_csv(PROCESSED/'21_consensus_tail_thresholds.csv')
n21_consensus_comparators = n21_consensus_comparators.loc[n21_consensus_comparators.alpha.eq(.05),['target','n_reference']].rename(columns={'n_reference':'n21_validation_consensus_reference'})

audit=[]
for stage, days in [('B',B),('C',C)]:
    row={'block':stage,'raw_block_rows':len(days),'R_consensus_adequacy_threshold':736,'I_consensus_adequacy_threshold':720}
    for target in ('R','I'):
        row[f'{target}_usable_target_rows'] = int(ratios.loc[ratios.sample_split.eq(stage)&ratios.target.eq(target),'model_day'].nunique())
        ncons = int(cons_counts.loc[(cons_counts.sample_split.eq(stage))&(cons_counts.target.eq(target)),'n_consensus_reference'].iloc[0])
        comparator = int(n21_consensus_comparators.loc[n21_consensus_comparators.target.eq(target),'n21_validation_consensus_reference'].iloc[0])
        row[f'{target}_consensus_reference'] = ncons
        row[f'{target}_n21_validation_comparator'] = comparator
        row[f'{target}_ratio_to_n21_validation_comparator'] = ncons/comparator
        row[f'{target}_pass'] = ncons >= row[f'{target}_consensus_adequacy_threshold']
        for model in FLEXIBLE_MODELS:
            count = int(model_counts.loc[(model_counts.sample_split.eq(stage))&(model_counts.target.eq(target))&(model_counts.model.eq(model)),'n_model_reference'].iloc[0])
            row[f'{target}_{model}_model_reference'] = count
    row['overall_pass'] = bool(row['R_pass'] and row['I_pass'])
    audit.append(row)
reference_adequacy_audit = pd.DataFrame(audit)

display(model_counts.sort_values(['sample_split','target','model']))
display(reference_adequacy_audit)
Bpass = bool(reference_adequacy_audit.loc[reference_adequacy_audit.block.eq('B'),'overall_pass'].iloc[0])
Cpass = bool(reference_adequacy_audit.loc[reference_adequacy_audit.block.eq('C'),'overall_pass'].iloc[0])
N22_REFERENCE_ADEQUACY = 'PASS' if Cpass else 'FAIL_STOP'
if not Cpass:
    print('N22 REFERENCE ADEQUACY: FAIL — STATE CONSTRUCTION STOPPED')
    print('No block boundaries were changed; no downstream application work was started.')
    raise RuntimeError('C failed the predeclared binding adequacy rule.')
BRANCH = 'PRIMARY_C_PLUS_VALIDATION' if Bpass else 'FALLBACK_VALIDATION_ONLY'
print('APPLICATION_HISTORY_BRANCH =',BRANCH)
if not Bpass:
    print('WARNING: fallback branch materially reduces pre-TEST application-state history; no future PnL was inspected.')


,sample_split,target,model,n_model_reference,n21_validation_model_reference,ratio_to_n21_validation_comparator
0,B,I,GB,913,945,0.966138
1,B,I,MLP,741,899,0.824249
2,B,I,RF,913,945,0.966138
3,B,I,TRANSFORMER,734,888,0.826577
4,B,R,GB,748,907,0.824697
5,B,R,MLP,755,919,0.821545
6,B,R,RF,755,919,0.821545
7,B,R,TRANSFORMER,755,919,0.821545
8,C,I,GB,932,945,0.986243
9,C,I,MLP,836,899,0.929922


,block,raw_block_rows,R_consensus_adequacy_threshold,I_consensus_adequacy_threshold,R_usable_target_rows,R_consensus_reference,R_n21_validation_comparator,R_ratio_to_n21_validation_comparator,R_pass,R_RF_model_reference,...,I_usable_target_rows,I_consensus_reference,I_n21_validation_comparator,I_ratio_to_n21_validation_comparator,I_pass,I_RF_model_reference,I_GB_model_reference,I_MLP_model_reference,I_TRANSFORMER_model_reference,overall_pass
0,B,1209,736,720,995,755,919,0.821545,True,755,...,1206,741,899,0.824249,True,913,913,741,734,True
1,C,1209,736,720,1141,877,919,0.954298,True,877,...,1209,836,899,0.929922,True,932,932,836,813,True


APPLICATION_HISTORY_BRANCH = PRIMARY_C_PLUS_VALIDATION


## 22.9–22.15 Frozen empirical-tail detector, prospective stages and episodes

In [5]:
# 22.9-22.15: frozen N21 detector mechanics, expanded for audit readability.
def empirical_two_sided_tail_score(u):
    return np.minimum(1.0, 2.0 * np.minimum(u, 1.0 - u))

def build_model_level_scores(reference_block, target_block):
    reference = ratios.loc[ratios.sample_split.eq(reference_block)].copy()
    target = ratios.loc[ratios.sample_split.eq(target_block)].copy()
    scored_parts = []
    for (target_name, model), part in pd.concat([reference, target], ignore_index=True).groupby(['target', 'model'], sort=True):
        reference_log = reference.loc[
            reference.target.eq(target_name) & reference.model.eq(model) & reference.ordinary_flag & reference.log_eligible,
            'log_omega'
        ].sort_values().to_numpy(float)
        assert len(reference_log) > 0
        result = part.copy()
        eligible = result.log_eligible
        U = np.searchsorted(reference_log, result.loc[eligible, 'log_omega'].to_numpy(float), side='right') / len(reference_log)
        result.loc[eligible, 'U'] = U
        result.loc[eligible, 'p_emp'] = empirical_two_sided_tail_score(U)
        scored_parts.append(result)
    return pd.concat(scored_parts, ignore_index=True)

def build_consensus_scores(model_level_scores):
    rows = []
    eligible_rows = model_level_scores.loc[model_level_scores.p_emp.notna()]
    for (day, split, target_name), group in eligible_rows.groupby(['model_day', 'sample_split', 'target'], sort=True):
        if len(group) < MIN_MODEL_SUPPORT:
            continue
        floor = group.floor_activated.fillna(False).astype(bool)
        rows.append({
            'model_day': day, 'sample_split': split, 'target': target_name,
            'ordinary_reference': bool(group.ordinary_flag.all()),
            'n_model_support': int(len(group)),
            'P': float(group.p_emp.median()),
            'omega': float(group.omega.median()),
            'L': float(group.log_omega.median()),
            'n_floor': int(floor.sum()),
            'any_floor': bool(floor.any()),
        })
    return pd.DataFrame(rows)

def calibrate_consensus_threshold(consensus, reference_block):
    ref = consensus.loc[consensus.sample_split.eq(reference_block) & consensus.ordinary_reference]
    threshold = ref.groupby('target').P.quantile(PRIMARY_ALPHA).to_dict()
    counts = ref.groupby('target').size().to_dict()
    assert set(threshold) == {'R', 'I'} and set(counts) == {'R', 'I'}
    return threshold, counts

def attach_known_event_flags(pairs):
    result = pairs.copy()
    r_days = set(events.loc[events.target.eq('R'), 'target_model_day'])
    i_days = set(events.loc[events.target.eq('I'), 'target_model_day'])
    result['known_event_R'] = result.model_day.isin(r_days)
    result['known_event_I_prev'] = result.previous_model_day.isin(i_days)
    result['known_primary_event_pair'] = result.known_event_R | result.known_event_I_prev
    result['strict_hidden'] = result.candidate & ~result.known_primary_event_pair
    return result

def construct_response_coordinates(pairs):
    result = pairs.copy()
    # N21-compatible raw medians and transform-consistent companions are deliberately separate.
    result['omega_R_consensus'] = result['omega_R']
    result['omega_I_prev_consensus'] = result['omega_I_prev']
    result['omega_R_from_L'] = np.exp(result['L_R'])
    result['omega_I_prev_from_L'] = np.exp(result['L_I_prev'])
    result['Delta_L'] = result.L_R - result.L_I_prev
    result['common_log_response'] = (result.L_R + result.L_I_prev) / 2.0
    result['R_response_direction'] = np.select([result.L_R.gt(0), result.L_R.lt(0)], ['ABOVE', 'BELOW'], default='NEUTRAL')
    result['I_prev_response_direction'] = np.select([result.L_I_prev.gt(0), result.L_I_prev.lt(0)], ['ABOVE', 'BELOW'], default='NEUTRAL')
    result['any_floor_pair'] = result.any_floor_R | result.any_floor_I_prev
    result['hidden_state_information_timing'] = 'AVAILABLE_AFTER_MODEL_DAY_T'
    return result

stage_consensus_cache = {}
def score_stage(reference_block, target_block):
    model_scores = build_model_level_scores(reference_block, target_block)
    consensus = build_consensus_scores(model_scores)
    threshold, reference_counts = calibrate_consensus_threshold(consensus, reference_block)
    stage_consensus_cache[(reference_block, target_block)] = consensus.copy()

    r_scores = consensus.loc[consensus.target.eq('R')].rename(columns={
        'P': 'P_R', 'omega': 'omega_R', 'L': 'L_R',
        'n_model_support': 'n_models_R', 'n_floor': 'n_floor_R', 'any_floor': 'any_floor_R'
    })
    i_scores = consensus.loc[consensus.target.eq('I')].rename(columns={
        'model_day': 'previous_model_day', 'sample_split': 'previous_sample_split',
        'P': 'P_I_prev', 'omega': 'omega_I_prev', 'L': 'L_I_prev',
        'n_model_support': 'n_models_I', 'n_floor': 'n_floor_I_prev', 'any_floor': 'any_floor_I_prev'
    })
    pairs = (
        panel[['model_day', 'previous_model_day', 'sample_split', 'previous_sample_split', 'canonical_index']]
        .merge(r_scores[['model_day', 'sample_split', 'P_R', 'omega_R', 'L_R', 'n_models_R', 'n_floor_R', 'any_floor_R']], on=['model_day', 'sample_split'], how='inner')
        .merge(i_scores[['previous_model_day', 'previous_sample_split', 'P_I_prev', 'omega_I_prev', 'L_I_prev', 'n_models_I', 'n_floor_I_prev', 'any_floor_I_prev']], on=['previous_model_day', 'previous_sample_split'], how='inner')
    )
    pairs = pairs.loc[pairs.sample_split.eq(pairs.previous_sample_split) & pairs.sample_split.eq(target_block)].copy()
    pairs['R_abnormal'] = pairs.P_R.le(threshold['R'])
    pairs['I_abnormal_prev'] = pairs.P_I_prev.le(threshold['I'])
    pairs['state'] = np.select(
        [pairs.R_abnormal & ~pairs.I_abnormal_prev, ~pairs.R_abnormal & pairs.I_abnormal_prev, pairs.R_abnormal & pairs.I_abnormal_prev],
        ['REALISED_ONLY', 'IMPLIED_ONLY', 'JOINT'],
        default='NONE'
    )
    pairs['candidate'] = pairs.state.ne('NONE')
    pairs['reference_block'] = reference_block
    pairs['stage'] = target_block
    pairs = attach_known_event_flags(pairs)
    pairs = construct_response_coordinates(pairs)
    return pairs, threshold, reference_counts

def construct_episodes(states, prefix):
    days = states.loc[states.strict_hidden].sort_values('canonical_index').copy()
    active_R = np.where(days.R_abnormal, days.P_R, np.nan)
    active_I = np.where(days.I_abnormal_prev, days.P_I_prev, np.nan)
    days['candidate_severity_tail_p'] = np.nanmin(np.column_stack([active_R, active_I]), axis=1) if len(days) else np.array([])
    days['new_episode'] = days.state.ne(days.state.shift()) | days.canonical_index.diff().ne(1)
    days['episode_number'] = days.new_episode.cumsum().astype(int)
    days['episode_id'] = prefix + days.episode_number.astype(str).str.zfill(3)
    days['episode_entry'] = days.episode_id.ne(days.episode_id.shift())
    rows = []
    for episode_id, group in days.groupby('episode_id', sort=True):
        peak = group.sort_values(['candidate_severity_tail_p', 'canonical_index']).iloc[0]
        rows.append({
            'episode_id': episode_id, 'stage': peak.stage, 'state': peak.state,
            'start_day': group.model_day.min(), 'end_day': group.model_day.max(),
            'peak_day': peak.model_day, 'peak_tail_p': float(peak.candidate_severity_tail_p),
            'number_of_days': int(len(group))
        })
    return days, pd.DataFrame(rows)

# B remains reference-only.  The passed decision gate permits C, VALIDATION, and TEST states.
if Bpass:
    Cstates, Cth, Cref = score_stage('B', 'C')
    Cepisodes, Cepisode_summary = construct_episodes(Cstates, 'N22_C_')
Vstates, Vth, Vref = score_stage('C', 'validation')
Vepisodes, Vepisode_summary = construct_episodes(Vstates, 'N22_V_')
Tstates, Tth, Tref = score_stage('validation', 'test')
Tepisodes, Tepisode_summary = construct_episodes(Tstates, 'N22_T_')

prospective = pd.concat(([Cstates] if Bpass else []) + [Vstates, Tstates], ignore_index=True)
eps = pd.concat(([Cepisodes] if Bpass else []) + [Vepisodes, Tepisodes], ignore_index=True)
prospective_episodes = pd.concat(([Cepisode_summary] if Bpass else []) + [Vepisode_summary, Tepisode_summary], ignore_index=True)
entries = eps.loc[eps.episode_entry].copy()
prospective = prospective.merge(eps[['model_day', 'episode_id', 'episode_entry']], on='model_day', how='left', validate='one_to_one')
prospective['episode_id'] = prospective.episode_id.fillna('')
prospective['episode_entry'] = prospective.episode_entry.astype('boolean').fillna(False).astype(bool)

stage_thresholds = pd.DataFrame(
    ([{'reference_block': 'B', 'target_stage': 'C', 'n_R_consensus_reference': Cref['R'], 'n_I_consensus_reference': Cref['I'], 'c_R_05': Cth['R'], 'c_I_05': Cth['I']}] if Bpass else [])
    + [
        {'reference_block': 'C', 'target_stage': 'VALIDATION', 'n_R_consensus_reference': Vref['R'], 'n_I_consensus_reference': Vref['I'], 'c_R_05': Vth['R'], 'c_I_05': Vth['I']},
        {'reference_block': 'VALIDATION', 'target_stage': 'TEST', 'n_R_consensus_reference': Tref['R'], 'n_I_consensus_reference': Tref['I'], 'c_R_05': Tth['R'], 'c_I_05': Tth['I']}
    ]
)
display(stage_thresholds)


,reference_block,target_stage,n_R_consensus_reference,n_I_consensus_reference,c_R_05,c_I_05
0,B,C,755,741,0.058063,0.066813
1,C,VALIDATION,877,836,0.055274,0.070606
2,VALIDATION,TEST,919,899,0.051034,0.062683


## 22.16 Diagnostics and TEST-equivalence setup

VALIDATION has two roles: N21 uses it as the ordinary calibration reference for TEST; N22 classifies it prospectively against C. TEST is independently reconstructed from the exact N20 inputs and reconciled to N21.

The broader empirical detector development had prior TEST exposure. From the final N21 freeze onward, TEST is not used here for application model, feature, cost, or policy selection. Episode entry is the first observed strict-hidden day; no contract logic, option PnL, or trading decision is performed.

## 22.17–22.18 Response-coordinate definitions and missing-run diagnostics

For each channel, N21 retains two different consensus summaries:

$$\omega_R^{raw,cons}=\operatorname{median}_m(\omega_{R,m}),\qquad L_R=\operatorname{median}_m[\log(\omega_{R,m})].$$

N22 therefore retains omega_R_consensus and adds omega_R_from_L = exp(L_R); the same definitions apply to previous-I. The detector and Delta_L = L_R-L_I use the log-response coordinates, while common_log_response=(L_R+L_I)/2.

The median raw response ratio and median log response are separate cross-model summaries. With even support, they are not exact transforms. The transform-consistent identity is:

$$\exp(\Delta L)=\frac{\omega_R^{fromL}}{\omega_{I,prev}^{fromL}},$$

not the ratio of the raw-median omega columns.

In [6]:
# 22.16: read-only N21 coordinate preflight, provenance, and descriptive surgical audits.
n21_panel = pd.read_csv(PROCESSED / '21_cross_channel_response_panel.csv', parse_dates=['model_day', 'previous_model_day'])
n21_test = n21_panel.loc[n21_panel.sample_split.eq('test')].copy()

# Frozen-N21 semantics preflight: raw medians and log medians are intentionally separate.
omega_audit_rows = []
support_rows = []
for channel, raw_column, log_column, support_column in [
    ('R', 'omega_R_consensus', 'L_R', 'n_models_R'),
    ('I_PREVIOUS', 'omega_I_prev_consensus', 'L_I_prev', 'n_models_I'),
]:
    part = n21_test.loc[
        np.isfinite(n21_test[raw_column]) & np.isfinite(n21_test[log_column]) & n21_test[raw_column].gt(0)
    ].copy()
    part['omega_from_L'] = np.exp(part[log_column])
    part['absolute_discrepancy'] = (part[raw_column] - part.omega_from_L).abs()
    part['relative_discrepancy'] = part.absolute_discrepancy / part[raw_column].abs()
    for support, group in part.groupby(support_column, sort=True):
        equal = np.isclose(group[raw_column], group.omega_from_L, rtol=0.0, atol=1e-12)
        omega_audit_rows.append({
            'channel': channel, 'n_models': int(support), 'row_count': int(len(group)),
            'share_of_valid_rows': float(len(group) / len(part)),
            'raw_equals_omega_from_L_atol_1e_12': int(equal.sum()),
            'median_absolute_discrepancy': float(group.absolute_discrepancy.median()),
            'maximum_absolute_discrepancy': float(group.absolute_discrepancy.max()),
            'median_relative_discrepancy': float(group.relative_discrepancy.median()),
            'maximum_relative_discrepancy': float(group.relative_discrepancy.max()),
        })
        support_rows.append({'channel': channel, 'n_models': int(support), 'row_count': int(len(group)), 'share_of_valid_rows': float(len(group) / len(part))})
omega_consensus_definition_audit = pd.DataFrame(omega_audit_rows)
model_support_distribution = pd.DataFrame(support_rows)

response_coordinate_definitions = pd.DataFrame([
    {'field': 'omega_R_consensus', 'definition': 'median_m(omega_R_m)', 'space': 'raw response ratio', 'detector_role': 'N21-compatible descriptive summary'},
    {'field': 'L_R / log_omega_R_consensus', 'definition': 'median_m(log(omega_R_m))', 'space': 'log response ratio', 'detector_role': 'detector coordinate'},
    {'field': 'omega_R_from_L', 'definition': 'exp(L_R)', 'space': 'raw transform of log coordinate', 'detector_role': 'transform-consistent descriptive companion'},
    {'field': 'omega_I_prev_consensus', 'definition': 'median_m(omega_I_prev_m)', 'space': 'raw response ratio', 'detector_role': 'N21-compatible descriptive summary'},
    {'field': 'L_I_prev / log_omega_I_prev_consensus', 'definition': 'median_m(log(omega_I_prev_m))', 'space': 'log response ratio', 'detector_role': 'detector coordinate'},
    {'field': 'omega_I_prev_from_L', 'definition': 'exp(L_I_prev)', 'space': 'raw transform of log coordinate', 'detector_role': 'transform-consistent descriptive companion'},
    {'field': 'Delta_L', 'definition': 'L_R - L_I_prev; exp(Delta_L)=omega_R_from_L/omega_I_prev_from_L', 'space': 'log difference', 'detector_role': 'detector-derived coordinate'},
    {'field': 'common_log_response', 'definition': '(L_R + L_I_prev)/2', 'space': 'log response ratio', 'detector_role': 'descriptive coordinate'},
])

# Row-level response-coordinate identities across all N22 application rows.
coordinate_rows = []
for stage, group in prospective.groupby('stage', sort=True):
    valid = group.loc[np.isfinite(group.L_R) & np.isfinite(group.L_I_prev) & np.isfinite(group.Delta_L) & np.isfinite(group.common_log_response) & group.omega_R_from_L.gt(0) & group.omega_I_prev_from_L.gt(0)].copy()
    residuals = {
        'L_R_equals_common_plus_half_delta': valid.L_R - (valid.common_log_response + valid.Delta_L / 2.0),
        'L_I_equals_common_minus_half_delta': valid.L_I_prev - (valid.common_log_response - valid.Delta_L / 2.0),
        'omega_R_from_L_equals_exp_L_R': valid.omega_R_from_L - np.exp(valid.L_R),
        'omega_I_from_L_equals_exp_L_I': valid.omega_I_prev_from_L - np.exp(valid.L_I_prev),
        'exp_delta_equals_from_L_ratio': np.exp(valid.Delta_L) - valid.omega_R_from_L / valid.omega_I_prev_from_L,
    }
    for identity, residual in residuals.items():
        residual = np.asarray(residual, dtype=float)
        passed = np.isclose(residual, 0.0, rtol=0.0, atol=1e-12)
        coordinate_rows.append({'scope': stage, 'identity': identity, 'n_comparable_rows': int(len(residual)), 'n_failing': int((~passed).sum()), 'maximum_absolute_residual': float(np.abs(residual).max()) if len(residual) else np.nan, 'pass': bool(passed.all())})
response_coordinate_identity_audit = pd.DataFrame(coordinate_rows)
# Continuous TEST reconciliation compares N21-compatible definitions only.
continuous_rows = []
n22_test = Tstates.set_index('model_day')
n21_test_indexed = n21_test.set_index('model_day')
for label, n22_column, n21_column in [
    ('L_R', 'L_R', 'L_R'),
    ('L_I_prev', 'L_I_prev', 'L_I_prev'),
    ('Delta_L', 'Delta_L', 'Delta_L'),
    ('common_log_response', 'common_log_response', 'common_log_response'),
    ('omega_R_consensus', 'omega_R_consensus', 'omega_R_consensus'),
    ('omega_I_prev_consensus', 'omega_I_prev_consensus', 'omega_I_prev_consensus'),
]:
    joined = pd.concat([n22_test[n22_column].rename('n22'), n21_test_indexed[n21_column].rename('n21')], axis=1, join='inner').dropna()
    diff = (joined.n22 - joined.n21).abs()
    continuous_rows.append({'object': label, 'n_comparable_rows': int(len(joined)), 'maximum_absolute_difference': float(diff.max()) if len(diff) else np.nan, 'n_failing_atol_1e_12': int((~np.isclose(joined.n22, joined.n21, rtol=0.0, atol=1e-12)).sum()), 'pass': bool(np.isclose(joined.n22, joined.n21, rtol=0.0, atol=1e-12).all())})
test_continuous_reconciliation = pd.DataFrame(continuous_rows)

# Floor provenance is descriptive only and never filters a state.
floor_rows = []
for stage, states, episode_days in [('C', Cstates, Cepisodes), ('VALIDATION', Vstates, Vepisodes), ('TEST', Tstates, Tepisodes)]:
    floor_rows.append({
        'stage': stage, 'valid_consensus_days': int(len(states)),
        'candidate_days_any_floor': int((states.candidate & states.any_floor_pair).sum()),
        'strict_hidden_days_any_floor': int((states.strict_hidden & states.any_floor_pair).sum()),
        'episode_entries_any_floor': int((episode_days.episode_entry & episode_days.any_floor_pair).sum()),
    })
floor_provenance_summary = pd.DataFrame(floor_rows)

# Attrition follows actual available fields; model-level quantities are explicitly counted as model rows.
stage_day_map = {'B': B, 'C': C, 'VALIDATION': V, 'TEST': T}
stage_ratio_map = {'B': 'B', 'C': 'C', 'VALIDATION': 'validation', 'TEST': 'test'}
attrition_rows = []
for stage, days in stage_day_map.items():
    for target in ['R', 'I']:
        ratio_part = ratios.loc[(ratios.sample_split.eq(stage_ratio_map[stage])) & ratios.target.eq(target)].copy()
        panel_part = panel.loc[panel.model_day.isin(days)].copy()
        consensus = stage_consensus_cache.get(('B', 'C') if stage in ['B', 'C'] else ('C', 'validation') if stage == 'VALIDATION' else ('validation', 'test'), pd.DataFrame())
        target_consensus = consensus.loc[consensus.target.eq(target) & consensus.sample_split.eq(stage_ratio_map[stage])] if len(consensus) else pd.DataFrame()
        ordinary_count = np.nan
        if stage in ['B', 'C']:
            ordinary_count = int(cons_counts.loc[(cons_counts.sample_split.eq(stage)) & (cons_counts.target.eq(target)), 'n_consensus_reference'].iloc[0])
        attrition_rows.append({
            'stage': stage, 'target': target, 'raw_canonical_model_days': int(len(days)),
            'finite_target_rows': int(np.isfinite(panel_part[target]).sum()),
            'model_forecast_rows_finite': int((np.isfinite(ratio_part.raw_forecast) & np.isfinite(ratio_part.actual)).sum()),
            'admissible_forecast_rows_positive': int((np.isfinite(ratio_part.admissible_forecast) & ratio_part.admissible_forecast.gt(0)).sum()),
            'positive_log_response_model_rows': int(ratio_part.log_eligible.sum()),
            'minimum_model_support_target_days': int(len(target_consensus)),
            'valid_target_consensus_days': int(len(target_consensus)),
            'ordinary_reference_eligible_consensus_days': ordinary_count,
            'raw_minus_finite_target': int(len(days) - np.isfinite(panel_part[target]).sum()),
            'finite_target_without_positive_model_log_rows': int(np.isfinite(panel_part[target]).sum() - ratio_part.loc[ratio_part.log_eligible, 'model_day'].nunique()),
        })
block_target_attrition_audit = pd.DataFrame(attrition_rows)

# Pair-level event audit uses target-specific N18 dates and set-based joins, so duplicate mapping rows cannot inflate pair totals.
event_rows = []
for stage, states in [('C', Cstates), ('VALIDATION', Vstates), ('TEST', Tstates)]:
    stage_union = pd.Series(False, index=states.index)
    for family in ['FOMC', 'US Employment', 'US CPI', 'BoJ', 'JPY National CPI', 'JPY GDP']:
        r_days = set(events.loc[events.target.eq('R') & events.family.eq(family), 'target_model_day'])
        i_days = set(events.loc[events.target.eq('I') & events.family.eq(family), 'target_model_day'])
        r_leg = states.model_day.isin(r_days)
        i_leg = states.previous_model_day.isin(i_days)
        union = r_leg | i_leg
        stage_union = stage_union | union
        event_rows.append({'stage': stage, 'family': family, 'n_R_event_legs': int(r_leg.sum()), 'n_previous_I_event_legs': int(i_leg.sum()), 'n_pair_or_for_family': int(union.sum()), 'unique_pair_join': bool(states.model_day.is_unique)})
    event_rows.append({'stage': stage, 'family': 'ALL_SIX', 'n_R_event_legs': int(states.known_event_R.sum()), 'n_previous_I_event_legs': int(states.known_event_I_prev.sum()), 'n_pair_or_for_family': int(stage_union.sum()), 'unique_pair_join': bool(states.model_day.is_unique)})
known_event_pair_audit = pd.DataFrame(event_rows)
assert known_event_pair_audit.loc[known_event_pair_audit.family.eq('ALL_SIX'), 'n_pair_or_for_family'].to_list() == [int(Cstates.known_primary_event_pair.sum()), int(Vstates.known_primary_event_pair.sum()), int(Tstates.known_primary_event_pair.sum())]
display(omega_consensus_definition_audit)
display(model_support_distribution)
display(response_coordinate_definitions)
display(floor_provenance_summary)
display(block_target_attrition_audit)
display(known_event_pair_audit)


,channel,n_models,row_count,share_of_valid_rows,raw_equals_omega_from_L_atol_1e_12,median_absolute_discrepancy,maximum_absolute_discrepancy,median_relative_discrepancy,maximum_relative_discrepancy
0,R,4,1132,1.000000,0,0.000134,0.024744,0.000181,0.035108
1,I_PREVIOUS,3,10,0.008834,10,0.000000,0.000000,0.000000,0.000000
2,I_PREVIOUS,4,1122,0.991166,0,0.000265,0.069375,0.000295,0.031847


,channel,n_models,row_count,share_of_valid_rows
0,R,4,1132,1.000000
1,I_PREVIOUS,3,10,0.008834
2,I_PREVIOUS,4,1122,0.991166


,field,definition,space,detector_role
0,omega_R_consensus,median_m(omega_R_m),raw response ratio,N21-compatible descriptive summary
1,L_R / log_omega_R_consensus,median_m(log(omega_R_m)),log response ratio,detector coordinate
2,omega_R_from_L,exp(L_R),raw transform of log coordinate,transform-consistent descriptive companion
3,omega_I_prev_consensus,median_m(omega_I_prev_m),raw response ratio,N21-compatible descriptive summary
4,L_I_prev / log_omega_I_prev_consensus,median_m(log(omega_I_prev_m)),log response ratio,detector coordinate
5,omega_I_prev_from_L,exp(L_I_prev),raw transform of log coordinate,transform-consistent descriptive companion
6,Delta_L,L_R - L_I_prev; exp(Delta_L)=omega_R_from_L/om...,log difference,detector-derived coordinate
7,common_log_response,(L_R + L_I_prev)/2,log response ratio,descriptive coordinate


,stage,valid_consensus_days,candidate_days_any_floor,strict_hidden_days_any_floor,episode_entries_any_floor
0,C,1029,1,1,1
1,VALIDATION,1133,1,1,1
2,TEST,1132,0,0,0


,stage,target,raw_canonical_model_days,finite_target_rows,model_forecast_rows_finite,admissible_forecast_rows_positive,positive_log_response_model_rows,minimum_model_support_target_days,valid_target_consensus_days,ordinary_reference_eligible_consensus_days,raw_minus_finite_target,finite_target_without_positive_model_log_rows
0,B,R,1209,1005,3970,3970,3970,995,995,755.0,204,10
1,B,I,1209,1208,4354,4354,4354,975,975,741.0,1,2
2,C,R,1209,1172,4536,4536,4536,1141,1141,877.0,37,31
3,C,I,1209,1209,4557,4557,4557,1084,1084,836.0,0,0
4,VALIDATION,R,1209,1195,4711,4711,4711,1181,1181,NaN,14,14
5,VALIDATION,I,1209,1209,4716,4716,4716,1155,1155,NaN,0,0
6,TEST,R,1209,1195,4715,4715,4715,1182,1182,NaN,14,13
7,TEST,I,1209,1208,4714,4714,4714,1156,1156,NaN,1,0


,stage,family,n_R_event_legs,n_previous_I_event_legs,n_pair_or_for_family,unique_pair_join
0,C,FOMC,33,34,67,True
1,C,US Employment,44,44,44,True
2,C,US CPI,51,51,51,True
3,C,BoJ,52,52,52,True
4,C,JPY National CPI,48,48,48,True
5,C,JPY GDP,32,32,32,True
6,C,ALL_SIX,240,240,266,True
7,VALIDATION,FOMC,36,35,71,True
8,VALIDATION,US Employment,51,51,51,True
9,VALIDATION,US CPI,55,55,55,True


In [7]:
# 22.18 B realised-variance missing-run diagnostic and timing implementation note.
B_missing_R = panel.loc[panel.model_day.isin(B) & ~np.isfinite(panel.R), ['model_day', 'canonical_index']].sort_values('canonical_index').copy()
B_missing_R['new_run'] = B_missing_R.canonical_index.diff().ne(1)
B_missing_R['run_number'] = B_missing_R.new_run.cumsum().astype(int)
B_missing_R['run_id'] = 'B_R_MISSING_' + B_missing_R.run_number.astype(str).str.zfill(3)

B_realised_missing_run_audit = (
    B_missing_R.groupby('run_id', as_index=False)
    .agg(start_model_day=('model_day', 'min'), end_model_day=('model_day', 'max'), n_missing_model_days=('model_day', 'size'))
    .sort_values(['n_missing_model_days', 'start_model_day'], ascending=[False, True])
    .reset_index(drop=True)
)
longest = B_realised_missing_run_audit.iloc[0]
B_realised_missing_run_summary = pd.DataFrame([{
    'total_B_missing_R_days': int(len(B_missing_R)),
    'number_of_missing_runs': int(len(B_realised_missing_run_audit)),
    'longest_missing_run_days': int(longest.n_missing_model_days),
    'median_missing_run_length': float(B_realised_missing_run_audit.n_missing_model_days.median()),
    'longest_run_start_model_day': longest.start_model_day,
    'longest_run_end_model_day': longest.end_model_day,
    'longest_run_missing_day_share': float(longest.n_missing_model_days / len(B_missing_R)),
}])
concentration = 'concentrated' if float(longest.n_missing_model_days / len(B_missing_R)) >= 0.5 else 'scattered across multiple canonical runs'
display(B_realised_missing_run_audit)
display(B_realised_missing_run_summary)
display(Markdown(
    f"The reduced B realised-variance coverage is **{concentration}**: {len(B_missing_R)} missing canonical model days occur in "
    f"{len(B_realised_missing_run_audit)} runs; the longest spans {int(longest.n_missing_model_days)} rows from "
    f"{pd.Timestamp(longest.start_model_day).date()} to {pd.Timestamp(longest.end_model_day).date()}. "
    "This is descriptive only and does not alter B adequacy."
))
display(Markdown(
    "Timing implementation note: under the frozen N18 contract, FOMC implied timing is I_D while the other five families use I_(D-1). "
    "With the N21 (R_t, I_(t-1)) pair this mechanically permits different/adjacent FOMC legs, whereas the non-FOMC mapping aligns R_D and I_(D-1) on the same pair. "
    "This is an implementation diagnostic, not a causal result."
))


,run_id,start_model_day,end_model_day,n_missing_model_days
0,B_R_MISSING_003,2010-01-01,2010-10-01,196
1,B_R_MISSING_001,2008-11-03,2008-11-03,1
2,B_R_MISSING_002,2009-11-02,2009-11-02,1
3,B_R_MISSING_004,2010-11-08,2010-11-08,1
4,B_R_MISSING_005,2011-11-07,2011-11-07,1
5,B_R_MISSING_006,2012-04-04,2012-04-04,1
6,B_R_MISSING_007,2012-04-09,2012-04-09,1
7,B_R_MISSING_008,2012-07-26,2012-07-26,1
8,B_R_MISSING_009,2012-08-02,2012-08-02,1


,total_B_missing_R_days,number_of_missing_runs,longest_missing_run_days,median_missing_run_length,longest_run_start_model_day,longest_run_end_model_day,longest_run_missing_day_share
0,204,9,196,1.0,2010-01-01,2010-10-01,0.960784


The reduced B realised-variance coverage is **concentrated**: 204 missing canonical model days occur in 9 runs; the longest spans 196 rows from 2010-01-01 to 2010-10-01. This is descriptive only and does not alter B adequacy.

Timing implementation note: under the frozen N18 contract, FOMC implied timing is I_D while the other five families use I_(D-1). With the N21 (R_t, I_(t-1)) pair this mechanically permits different/adjacent FOMC legs, whereas the non-FOMC mapping aligns R_D and I_(D-1) on the same pair. This is an implementation diagnostic, not a causal result.

## 22.19 Historical C forecasts, TEST reconciliation, and canonical exports

This export exposes the exact C model-level ordinary forecasts already computed by N22's frozen `ratios` object. It does not alter forecasting, calibration, detection, state classification, episode construction, or any existing scientific result. The forecasts belong to the retrospective chronological reconstruction conditional on the final fixed specification: C parameters use A+B only, while frozen architectures/specifications used broader pre-TEST information. They are not presented as prospective or live forecasts.

In [8]:
# 22.19 Exact TEST reconciliation, stage diagnostics, and canonical exports.
# 1. Load the frozen N21 TEST panel, strict-hidden dates, episodes, and thresholds.
canon_pairs = pd.read_csv(PROCESSED / '21_cross_channel_response_panel.csv', parse_dates=['model_day'])
canon_strict = pd.read_csv(PROCESSED / '21_candidate_days_primary.csv', parse_dates=['model_day'])
canon_eps = pd.read_csv(PROCESSED / '21_hidden_episodes_primary.csv', parse_dates=['start_day', 'end_day', 'peak_day'])
canonical_thresholds = pd.read_csv(PROCESSED / '21_consensus_tail_thresholds.csv')

# 2. Read canonical primary thresholds and prepare the independently reconstructed N22 TEST pairs.
canonical_c_R = float(canonical_thresholds.loc[canonical_thresholds.target.eq('R') & canonical_thresholds.alpha.eq(.05), 'threshold'].iloc[0])
canonical_c_I = float(canonical_thresholds.loc[canonical_thresholds.target.eq('I') & canonical_thresholds.alpha.eq(.05), 'threshold'].iloc[0])
test_pairs = Tstates.sort_values('model_day').copy()
canonical_test = canon_pairs.loc[canon_pairs.sample_split.eq('test')].sort_values('model_day').copy()

# 3. Exact discrete reconciliation: candidate dates, strict-hidden dates, states, and episodes.
candidate_date_set_match = set(test_pairs.loc[test_pairs.candidate, 'model_day']) == set(canonical_test.loc[canonical_test.is_candidate, 'model_day'])
strict_hidden_date_set_match = set(test_pairs.loc[test_pairs.strict_hidden, 'model_day']) == set(canon_strict.model_day)
state_label_match = test_pairs.set_index('model_day').state.equals(
    canonical_test.set_index('model_day').primary_state.reindex(test_pairs.model_day)
)

canonical_episode_chronology = (
    canon_eps[['primary_state', 'start_day', 'end_day', 'peak_day', 'number_of_days']]
    .rename(columns={'primary_state': 'state'})
    .sort_values(['start_day', 'end_day'])
    .reset_index(drop=True)
)
n22_episode_chronology = (
    Tepisode_summary[['state', 'start_day', 'end_day', 'peak_day', 'number_of_days']]
    .sort_values(['start_day', 'end_day'])
    .reset_index(drop=True)
)
episode_chronology_match = canonical_episode_chronology.equals(n22_episode_chronology)

# 4. Threshold differences use the frozen 1e-12 tolerance, not rounded display values.
recon = pd.DataFrame([{
    'canonical_c_R': canonical_c_R,
    'reconstructed_c_R': Tth['R'],
    'abs_diff_R': abs(canonical_c_R - Tth['R']),
    'R_tolerance_pass': np.isclose(canonical_c_R, Tth['R'], rtol=0.0, atol=1e-12),
    'canonical_c_I': canonical_c_I,
    'reconstructed_c_I': Tth['I'],
    'abs_diff_I': abs(canonical_c_I - Tth['I']),
    'I_tolerance_pass': np.isclose(canonical_c_I, Tth['I'], rtol=0.0, atol=1e-12),
    'canonical_candidate_count': 209,
    'reconstructed_candidate_count': int(test_pairs.candidate.sum()),
    'candidate_date_set_match': candidate_date_set_match,
    'canonical_strict_hidden_days': 93,
    'reconstructed_strict_hidden_days': int(test_pairs.strict_hidden.sum()),
    'strict_hidden_date_set_match': strict_hidden_date_set_match,
    'canonical_episode_count': 85,
    'reconstructed_episode_count': int(Tepisodes.episode_id.nunique()),
    'state_label_match': state_label_match,
    'episode_chronology_match': episode_chronology_match,
}])
recon['overall_reconciliation_status'] = recon[
    ['R_tolerance_pass', 'I_tolerance_pass', 'candidate_date_set_match', 'strict_hidden_date_set_match', 'state_label_match', 'episode_chronology_match']
].all(axis=1)

# 5. Stage diagnostics distinguish complete target-block length from valid paired consensus rows.
def stage_summary_row(stage, states, episode_days, raw_target_rows):
    return {
        'stage': stage,
        'raw_target_rows': int(raw_target_rows),
        'valid_consensus_pairs': int(len(states)),
        'R_abnormal_count': int(states.R_abnormal.sum()),
        'R_abnormal_rate': float(states.R_abnormal.mean()),
        'I_abnormal_count': int(states.I_abnormal_prev.sum()),
        'I_abnormal_rate': float(states.I_abnormal_prev.mean()),
        'candidate_count': int(states.candidate.sum()),
        'NONE': int(states.state.eq('NONE').sum()),
        'REALISED_ONLY': int(states.state.eq('REALISED_ONLY').sum()),
        'IMPLIED_ONLY': int(states.state.eq('IMPLIED_ONLY').sum()),
        'JOINT': int(states.state.eq('JOINT').sum()),
        'known_primary_event_pair_count': int(states.known_primary_event_pair.sum()),
        'strict_hidden_days': int(states.strict_hidden.sum()),
        'strict_hidden_episodes': int(episode_days.episode_id.nunique()),
        'episode_entries': int(episode_days.episode_entry.sum()),
    }

stage_summary = pd.DataFrame([
    stage_summary_row('C', Cstates, Cepisodes, len(C)),
    stage_summary_row('VALIDATION', Vstates, Vepisodes, len(V)),
    stage_summary_row('TEST', Tstates, Tepisodes, len(T)),
])

# 6. N21 VALIDATION reference and N22 prospective VALIDATION have intentionally different roles.
n21_reference_counts = canonical_thresholds.loc[canonical_thresholds.alpha.eq(.05)].set_index('target').n_reference.to_dict()
dual = pd.DataFrame([
    {
        'role': 'N21_REFERENCE_DIAGNOSTIC', 'reference_block': 'VALIDATION',
        'n_R_consensus_reference': n21_reference_counts['R'],
        'n_I_consensus_reference': n21_reference_counts['I'],
        'c_R': canonical_c_R, 'c_I': canonical_c_I,
        'R_abnormal_count': np.nan, 'I_abnormal_count': np.nan, 'candidate_count': np.nan,
    },
    {
        'role': 'N22_PROSPECTIVE_VALIDATION', 'reference_block': 'C',
        'n_R_consensus_reference': int(cons_counts.loc[cons_counts.sample_split.eq('C') & cons_counts.target.eq('R'), 'n_consensus_reference'].iloc[0]),
        'n_I_consensus_reference': int(cons_counts.loc[cons_counts.sample_split.eq('C') & cons_counts.target.eq('I'), 'n_consensus_reference'].iloc[0]),
        'c_R': Vth['R'], 'c_I': Vth['I'],
        'R_abnormal_count': int(Vstates.R_abnormal.sum()),
        'I_abnormal_count': int(Vstates.I_abnormal_prev.sum()),
        'candidate_count': int(Vstates.candidate.sum()),
    },
])

stage_thresholds = pd.DataFrame([
    {'reference_block': 'B', 'target_stage': 'C', 'n_R_consensus_reference': Cref['R'], 'n_I_consensus_reference': Cref['I'], 'c_R_05': Cth['R'], 'c_I_05': Cth['I']},
    {'reference_block': 'C', 'target_stage': 'VALIDATION', 'n_R_consensus_reference': Vref['R'], 'n_I_consensus_reference': Vref['I'], 'c_R_05': Vth['R'], 'c_I_05': Vth['I']},
    {'reference_block': 'VALIDATION', 'target_stage': 'TEST', 'n_R_consensus_reference': Tref['R'], 'n_I_consensus_reference': Tref['I'], 'c_R_05': Tth['R'], 'c_I_05': Tth['I']},
])

spec = pd.DataFrame([{
    'application_question': 'does_hidden_realised_implied_response_information_carry_incremental_economic_value',
    'notebook_role': 'leakage_safe_hidden_state_reconstruction',
    'primary_alpha': .05,
    'historical_reconstruction': 'pseudo_oos_conditional_on_final_frozen_specification',
    'forecast_estimation': 'expanding_history',
    'abnormality_reference': 'previous_frozen_oos_block',
    'candidate_A_start': '2003-05-05', 'candidate_A_end': '2007-12-19',
    'candidate_B_start': '2007-12-20', 'candidate_B_end': '2012-08-07',
    'candidate_C_start': '2012-08-08', 'candidate_C_end': '2017-03-27',
    'consensus_R_binding_min': 736, 'consensus_I_binding_min': 720,
    'B_self_classification': False,
    'episode_entry': 'first_observable_strict_hidden_state',
    'option_contract_logic_in_N22': False,
    'test_forecasts_refit': False,
    'test_forecast_source': 'canonical_N20_inputs_used_by_N21',
    'test_threshold_tolerance': 1e-12,
    'test_state_reconciliation': 'exact',
    'upstream_methodology_modified': False,
}])

# Export-only provenance amendment: exact model-level C rows already computed in `ratios`.
# These are pseudo-out-of-sample conditional on the final frozen specification:
# parameters for C use A+B only, while frozen architectures/specifications used broader pre-TEST information.
historical_C_model_level_ordinary_forecasts = ratios.loc[ratios.sample_split.eq('C')].copy()
historical_C_model_level_ordinary_forecasts['forecast_target_model_day'] = historical_C_model_level_ordinary_forecasts['model_day']
historical_C_model_level_ordinary_forecasts['fit_start_model_day'] = A.min()
historical_C_model_level_ordinary_forecasts['fit_end_model_day'] = B.max()
historical_C_model_level_ordinary_forecasts['forecast_fit_window'] = 'A_PLUS_B_TO_C'
assert historical_C_model_level_ordinary_forecasts.sample_split.eq('C').all()
assert not historical_C_model_level_ordinary_forecasts.duplicated(['sample_split', 'target', 'model', 'forecast_target_model_day']).any()

outputs = {
    '22_historical_C_model_level_ordinary_forecasts.csv': historical_C_model_level_ordinary_forecasts,
    '22_reference_adequacy_audit.csv': reference_adequacy_audit,
    '22_model_level_reference_counts.csv': model_counts,
    '22_stage_consensus_thresholds.csv': stage_thresholds,
    '22_stage_state_summary.csv': stage_summary,
    '22_validation_dual_role_audit.csv': dual,
    '22_prospective_episodes.csv': prospective_episodes,
    '22_episode_entries.csv': entries,
    '22_application_state_panel.csv': prospective,
    '22_prospective_episode_days.csv': eps,
    '22_test_reconciliation.csv': recon,
    '22_primary_specification.csv': spec,
    '22_omega_consensus_definition_audit.csv': omega_consensus_definition_audit,
    '22_model_support_distribution.csv': model_support_distribution,
    '22_response_coordinate_definitions.csv': response_coordinate_definitions,
    '22_response_coordinate_identity_audit.csv': response_coordinate_identity_audit,
    '22_test_continuous_reconciliation.csv': test_continuous_reconciliation,
    '22_floor_provenance_summary.csv': floor_provenance_summary,
    '22_block_target_attrition_audit.csv': block_target_attrition_audit,
    '22_known_event_pair_audit.csv': known_event_pair_audit,
    '22_B_realised_missing_run_audit.csv': B_realised_missing_run_audit,
    '22_B_realised_missing_run_summary.csv': B_realised_missing_run_summary,
}
for name, frame in outputs.items():
    frame.to_csv(PROCESSED / name, index=False)

manifest = pd.DataFrame([
    {'output_path': name, 'purpose': 'N22 leakage-safe reconstruction', 'branch': BRANCH, 'row_count': len(frame), 'required_nonempty': len(frame) > 0}
    for name, frame in outputs.items()
])
manifest.to_csv(PROCESSED / '22_export_manifest.csv', index=False)

display(stage_summary)
display(dual)
display(recon)
display(manifest)
assert bool(recon.loc[0, 'overall_reconciliation_status']), 'N22 TEST reconciliation failed.'
assert int(test_pairs.candidate.sum()) == 209
assert int(test_pairs.strict_hidden.sum()) == 93
assert int(Tepisodes.episode_id.nunique()) == 85


,stage,raw_target_rows,valid_consensus_pairs,R_abnormal_count,R_abnormal_rate,I_abnormal_count,I_abnormal_rate,candidate_count,NONE,REALISED_ONLY,IMPLIED_ONLY,JOINT,known_primary_event_pair_count,strict_hidden_days,strict_hidden_episodes,episode_entries
0,C,1209,1029,71,0.068999,83,0.080661,140,889,57,69,14,266,96,87,87
1,VALIDATION,1209,1133,38,0.033539,37,0.032657,70,1063,33,32,5,284,49,46,46
2,TEST,1209,1132,84,0.074205,144,0.127208,209,923,65,125,19,284,93,85,85


,role,reference_block,n_R_consensus_reference,n_I_consensus_reference,c_R,c_I,R_abnormal_count,I_abnormal_count,candidate_count
0,N21_REFERENCE_DIAGNOSTIC,VALIDATION,919,899,0.051034,0.062683,NaN,NaN,NaN
1,N22_PROSPECTIVE_VALIDATION,C,877,836,0.055274,0.070606,38.0,37.0,70.0


,canonical_c_R,reconstructed_c_R,abs_diff_R,R_tolerance_pass,canonical_c_I,reconstructed_c_I,abs_diff_I,I_tolerance_pass,canonical_candidate_count,reconstructed_candidate_count,candidate_date_set_match,canonical_strict_hidden_days,reconstructed_strict_hidden_days,strict_hidden_date_set_match,canonical_episode_count,reconstructed_episode_count,state_label_match,episode_chronology_match,overall_reconciliation_status
0,0.051034,0.051034,6.938894e-17,True,0.062683,0.062683,2.775558e-17,True,209,209,True,93,93,True,85,85,True,True,True


,output_path,purpose,branch,row_count,required_nonempty
0,22_historical_C_model_level_ordinary_forecasts...,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,9093,True
1,22_reference_adequacy_audit.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,2,True
2,22_model_level_reference_counts.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,16,True
3,22_stage_consensus_thresholds.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,3,True
4,22_stage_state_summary.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,3,True
5,22_validation_dual_role_audit.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,2,True
6,22_prospective_episodes.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,218,True
7,22_episode_entries.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,218,True
8,22_application_state_panel.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,3294,True
9,22_prospective_episode_days.csv,N22 leakage-safe reconstruction,PRIMARY_C_PLUS_VALIDATION,238,True


## 22.20–22.22 Surgical closure notes and invariance checks

**Documentation clarification to frozen N21.** N21 reports the raw-median consensus omega and the log-median consensus response separately. Delta_L is constructed from the log-median coordinates. The new omega_from_L columns expose the exact inverse transform of those coordinates; they do not redefine N21-compatible raw-median omega fields.

B's realised consensus ordinary-reference count is 755 against a binding minimum of 736: a modest +19-observation margin (755/919 of the final VALIDATION comparator). The primary branch is retained because this predeclared rule was satisfied, not because later state rates or application PnL were inspected.

Prospective firing rates vary across C, VALIDATION, and TEST despite similar target-specific thresholds. This documents temporal calibration-transfer/response-distribution heterogeneity, but does not attribute it uniquely to an underlying market-distribution change: historical stages also have different parameter fits and data coverage.

**N24 handoff only.** C is reserved for application-model development; VALIDATION for application specification selection; C+VALIDATION for the final pre-TEST refit after the specification freezes; TEST for economic evaluation. Hyperparameter selection must use predictive loss across all eligible complete VALIDATION contracts, not performance in the small hidden subset. C and VALIDATION must be reported separately.

In [9]:
# 22.22 Final surgical invariance and closure.
def baseline_threshold_value(frame, reference_block, coordinate):
    row = frame.loc[frame.reference_block.eq(reference_block)].iloc[0]
    for column in [coordinate, coordinate + '_05']:
        if column in frame.columns:
            return float(row[column])
    raise KeyError(coordinate)

invariance_rows = []
def add_invariance(check, before, after, kind='numeric'):
    passed = bool(np.isclose(float(before), float(after), rtol=0.0, atol=1e-12)) if kind == 'numeric' else bool(before == after)
    invariance_rows.append({'check': check, 'before': str(before), 'after': str(after), 'pass': passed})

pre_adequacy = pre_surgical_snapshot['adequacy']
for block, target, expected in [('B', 'R', 755), ('B', 'I', 741), ('C', 'R', None), ('C', 'I', None)]:
    before = int(pre_adequacy.loc[pre_adequacy.block.eq(block), f'{target}_consensus_reference'].iloc[0])
    after = int(reference_adequacy_audit.loc[reference_adequacy_audit.block.eq(block), f'{target}_consensus_reference'].iloc[0])
    add_invariance(f'{block}_{target}_consensus_reference', before, after, 'exact')
    if expected is not None:
        add_invariance(f'{block}_{target}_consensus_expected', expected, after, 'exact')

pre_thresholds = pre_surgical_snapshot['thresholds']
for reference_block, new_row in stage_thresholds.set_index('reference_block').iterrows():
    for coordinate in ['c_R', 'c_I']:
        add_invariance(f'{reference_block}_{coordinate}_threshold', baseline_threshold_value(pre_thresholds, reference_block, coordinate), float(new_row[coordinate + '_05']))

pre_stage = pre_surgical_snapshot['stage_summary']
for stage in ['C', 'VALIDATION', 'TEST']:
    before = pre_stage.loc[pre_stage.stage.eq(stage)].iloc[0]
    after = stage_summary.loc[stage_summary.stage.eq(stage)].iloc[0]
    for field in ['candidate_count', 'strict_hidden_days', 'REALISED_ONLY', 'IMPLIED_ONLY', 'JOINT', 'episode_entries']:
        add_invariance(f'{stage}_{field}', int(before[field]), int(after[field]), 'exact')

pre_application = pre_surgical_snapshot['application']
before_test = pre_application.loc[pre_application.stage.eq('test')].copy()
after_test = prospective.loc[prospective.stage.eq('test')].copy()
add_invariance('TEST_candidate_date_set', tuple(sorted(before_test.loc[before_test.candidate, 'model_day'].astype(str))), tuple(sorted(after_test.loc[after_test.candidate, 'model_day'].astype(str))), 'exact')
add_invariance('TEST_strict_hidden_date_set', tuple(sorted(before_test.loc[before_test.strict_hidden, 'model_day'].astype(str))), tuple(sorted(after_test.loc[after_test.strict_hidden, 'model_day'].astype(str))), 'exact')
add_invariance('TEST_state_labels', before_test.set_index('model_day').state.sort_index().astype(str).to_dict(), after_test.set_index('model_day').state.sort_index().astype(str).to_dict(), 'exact')

pre_episodes = pre_surgical_snapshot['episodes']
before_episode_chronology = pre_episodes.loc[pre_episodes.stage.eq('test'), ['state', 'start_day', 'end_day', 'peak_day', 'number_of_days']].sort_values(['start_day', 'end_day']).astype(str).to_dict('records')
after_episode_chronology = prospective_episodes.loc[prospective_episodes.stage.eq('test'), ['state', 'start_day', 'end_day', 'peak_day', 'number_of_days']].sort_values(['start_day', 'end_day']).astype(str).to_dict('records')
add_invariance('TEST_episode_chronology', before_episode_chronology, after_episode_chronology, 'exact')

surgical_invariance_audit = pd.DataFrame(invariance_rows)
assert surgical_invariance_audit['pass'].all(), 'Scientific invariance failed: ' + ', '.join(surgical_invariance_audit.loc[~surgical_invariance_audit['pass'], 'check'])
assert test_continuous_reconciliation['pass'].all(), 'Continuous N21 TEST reconciliation failed.'
assert response_coordinate_identity_audit['pass'].all(), 'Response-coordinate identities failed.'
assert int(B_realised_missing_run_summary.loc[0, 'total_B_missing_R_days']) == 204
assert len(B_realised_missing_run_audit) > 0
assert int(floor_provenance_summary.loc[floor_provenance_summary.stage.eq('TEST'), 'strict_hidden_days_any_floor'].iloc[0]) == 0
assert recon.loc[0, 'overall_reconciliation_status']
assert int(test_pairs.candidate.sum()) == 209 and int(test_pairs.strict_hidden.sum()) == 93 and int(Tepisodes.episode_id.nunique()) == 85

surgical_invariance_audit.to_csv(PROCESSED / '22_surgical_invariance_audit.csv', index=False)
manifest = pd.read_csv(PROCESSED / '22_export_manifest.csv')
manifest = pd.concat([manifest, pd.DataFrame([{'output_path': '22_surgical_invariance_audit.csv', 'purpose': 'N22 final surgical invariance audit', 'branch': BRANCH, 'row_count': len(surgical_invariance_audit), 'required_nonempty': len(surgical_invariance_audit) > 0}])], ignore_index=True)
assert manifest.output_path.is_unique
manifest.to_csv(PROCESSED / '22_export_manifest.csv', index=False)

display(test_continuous_reconciliation)
display(response_coordinate_identity_audit)
display(surgical_invariance_audit)
print('N22 FINAL FREEZE CLOSURE: PASS')
print('Reference branch: PRIMARY_C_PLUS_VALIDATION')
print('B consensus adequacy: R = 755 / 736; I = 741 / 720; realised margin = +19')
print('Prospective episode entries: C = 87; VALIDATION = 46; TEST = 85')
print('TEST detector: candidates = 209; strict-hidden days = 93; strict-hidden episodes = 85')
print('TEST date/state/episode reconciliation: PASS')
print('N21-compatible raw omega and log-response continuous reconciliation: PASS')
print('omega_from_L identities: PASS')
print('Floor provenance strict-hidden counts: ' + str(floor_provenance_summary.set_index('stage').strict_hidden_days_any_floor.to_dict()))
print('Raw target rows correction: PASS')
print('Event-pair audit: PASS')
print('Scientific invariance: PASS')
print('N18-N21 modified: NO')
print('Option contract logic: NOT STARTED; option PnL: NOT CALCULATED; application GB: NOT FIT')
print('Ready to freeze N22: YES')


,object,n_comparable_rows,maximum_absolute_difference,n_failing_atol_1e_12,pass
0,L_R,1132,4.440892e-16,0,True
1,L_I_prev,1132,2.220446e-16,0,True
2,Delta_L,1132,4.440892e-16,0,True
3,common_log_response,1132,2.220446e-16,0,True
4,omega_R_consensus,1132,1.776357e-15,0,True
5,omega_I_prev_consensus,1132,1.776357e-15,0,True


,scope,identity,n_comparable_rows,n_failing,maximum_absolute_residual,pass
0,C,L_R_equals_common_plus_half_delta,1029,0,2.220446e-16,True
1,C,L_I_equals_common_minus_half_delta,1029,0,2.220446e-16,True
2,C,omega_R_from_L_equals_exp_L_R,1029,0,0.000000e+00,True
3,C,omega_I_from_L_equals_exp_L_I,1029,0,0.000000e+00,True
4,C,exp_delta_equals_from_L_ratio,1029,0,8.881784e-16,True
5,test,L_R_equals_common_plus_half_delta,1132,0,2.220446e-16,True
6,test,L_I_equals_common_minus_half_delta,1132,0,1.942890e-16,True
7,test,omega_R_from_L_equals_exp_L_R,1132,0,0.000000e+00,True
8,test,omega_I_from_L_equals_exp_L_I,1132,0,0.000000e+00,True
9,test,exp_delta_equals_from_L_ratio,1132,0,7.105427e-15,True


,check,before,after,pass
0,B_R_consensus_reference,755,755,True
1,B_R_consensus_expected,755,755,True
2,B_I_consensus_reference,741,741,True
3,B_I_consensus_expected,741,741,True
4,C_R_consensus_reference,877,877,True
5,C_I_consensus_reference,836,836,True
6,B_c_R_threshold,0.0580630024436023,0.05806300244360233,True
7,B_c_I_threshold,0.0668127053669223,0.0668127053669223,True
8,C_c_R_threshold,0.0552735263729851,0.05527352637298515,True
9,C_c_I_threshold,0.0706059552150442,0.07060595521504427,True


N22 FINAL FREEZE CLOSURE: PASS
Reference branch: PRIMARY_C_PLUS_VALIDATION
B consensus adequacy: R = 755 / 736; I = 741 / 720; realised margin = +19
Prospective episode entries: C = 87; VALIDATION = 46; TEST = 85
TEST detector: candidates = 209; strict-hidden days = 93; strict-hidden episodes = 85
TEST date/state/episode reconciliation: PASS
N21-compatible raw omega and log-response continuous reconciliation: PASS
omega_from_L identities: PASS
Floor provenance strict-hidden counts: {'C': 1, 'VALIDATION': 1, 'TEST': 0}
Raw target rows correction: PASS
Event-pair audit: PASS
Scientific invariance: PASS
N18-N21 modified: NO
Option contract logic: NOT STARTED; option PnL: NOT CALCULATED; application GB: NOT FIT
Ready to freeze N22: YES
